In [1]:
import pandas as pd
import re

train = pd.read_json("D:/vuln-detection-ml/data/train.jsonl", lines=True)
valid = pd.read_json("D:/vuln-detection-ml/data/valid.jsonl", lines=True)
def clean_code(code):
    code = re.sub(r"/\*.*?\*/", " ", code, flags=re.DOTALL)  # blocks comments
    code = re.sub(r"//[^\n]*", " ", code)                    # line comments
    code = re.sub(r"\s+", " ", code)                         # collapse whitespace
    return code.strip()

train["clean"] = train["func"].apply(clean_code)
valid["clean"] = valid["func"].apply(clean_code)

print(train["clean"].iloc[0][:300])

bool virtio_scsi_handle_cmd_req_prepare(VirtIOSCSI *s, VirtIOSCSIReq *req) { VirtIOSCSICommon *vs = &s->parent_obj; SCSIDevice *d; int rc; rc = virtio_scsi_parse_req(req, sizeof(VirtIOSCSICmdReq) + vs->cdb_size, sizeof(VirtIOSCSICmdResp) + vs->sense_size); if (rc < 0) { if (rc == -ENOTSUP) { virtio_


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    token_pattern=r"[A-Za-z_][A-Za-z0-9_]*|\d+|==|!=|<=|>=|->|&&|\|\||[^\s\w]",
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000,
)

X_train = vectorizer.fit_transform(train["clean"])   # fit ONLY on train
X_valid = vectorizer.transform(valid["clean"])       # just transform here
y_train, y_valid = train["target"], valid["target"]

print(X_train.shape)

(21854, 50000)


In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Dummy accuracy:", dummy.score(X_valid, y_valid))

nb = MultinomialNB().fit(X_train, y_train)
pred = nb.predict(X_valid)
print(classification_report(y_valid, pred, digits=3))
print(confusion_matrix(y_valid, pred))

Dummy accuracy: 0.5439238653001464
              precision    recall  f1-score   support

           0      0.628     0.745     0.682      1486
           1      0.609     0.474     0.533      1246

    accuracy                          0.622      2732
   macro avg      0.619     0.610     0.608      2732
weighted avg      0.620     0.622     0.614      2732

[[1107  379]
 [ 655  591]]


In [4]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

for C in [0.01, 0.1, 1, 10]:
    svm = LinearSVC(C=C, max_iter=5000).fit(X_train, y_train)
    p = svm.predict(X_valid)
    print(f"C={C}: accuracy={accuracy_score(y_valid, p):.3f}, F1(vuln)={f1_score(y_valid, p):.3f}")

C=0.01: accuracy=0.598, F1(vuln)=0.432
C=0.1: accuracy=0.631, F1(vuln)=0.559
C=1: accuracy=0.631, F1(vuln)=0.584
C=10: accuracy=0.601, F1(vuln)=0.562


In [5]:
for alpha in [0.01, 0.1, 0.5, 1.0]:
    nb = MultinomialNB(alpha=alpha).fit(X_train, y_train)
    p = nb.predict(X_valid)
    print(f"alpha={alpha}: accuracy={accuracy_score(y_valid, p):.3f}, F1(vuln)={f1_score(y_valid, p):.3f}")

alpha=0.01: accuracy=0.630, F1(vuln)=0.587
alpha=0.1: accuracy=0.630, F1(vuln)=0.582
alpha=0.5: accuracy=0.627, F1(vuln)=0.562
alpha=1.0: accuracy=0.622, F1(vuln)=0.533


In [6]:
for alpha in [0.0001, 0.001, 0.01]:
    nb = MultinomialNB(alpha=alpha).fit(X_train, y_train)
    p = nb.predict(X_valid)
    print(f"NB alpha={alpha}: acc={accuracy_score(y_valid, p):.3f}, F1={f1_score(y_valid, p):.3f}")

for C in [0.5, 1]:
    svm = LinearSVC(C=C, class_weight="balanced", max_iter=5000).fit(X_train, y_train)
    p = svm.predict(X_valid)
    print(f"SVM balanced C={C}: acc={accuracy_score(y_valid, p):.3f}, F1={f1_score(y_valid, p):.3f}")

NB alpha=0.0001: acc=0.630, F1=0.587
NB alpha=0.001: acc=0.629, F1=0.587
NB alpha=0.01: acc=0.630, F1=0.587
SVM balanced C=0.5: acc=0.636, F1=0.611
SVM balanced C=1: acc=0.626, F1=0.601


In [8]:
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

svm = LinearSVC(C=0.5, class_weight="balanced", max_iter=5000).fit(X_train, y_train)
p = svm.predict(X_valid)
print(classification_report(y_valid, p, digits=3))
print(confusion_matrix(y_valid, p))

names = np.array(vectorizer.get_feature_names_out())
order = np.argsort(svm.coef_[0])
print("Pushes toward SAFE:", list(names[order[:20]]))
print("Pushes toward VULNERABLE:", list(names[order[-20:]]))

              precision    recall  f1-score   support

           0      0.673     0.643     0.658      1486
           1      0.596     0.628     0.611      1246

    accuracy                          0.636      2732
   macro avg      0.635     0.635     0.635      2732
weighted avg      0.638     0.636     0.637      2732

[[956 530]
 [464 782]]
Pushes toward SAFE: ['}', '; }', '} }', ') return', 'target_phys_addr_t', 'str +', '&& (', ', target_phys_addr_t', ') buf', 'target_phys_addr_t base', '( ch', 'config ,', 'ioc ,', 'srcp', ', __FUNCTION__', 'coded_frame )', 'src_wrap', 'coded_frame', '__FUNCTION__', 'buf_offset']
Pushes toward VULNERABLE: ['-> config_reg', 'config_reg', 'qsb', 'ahci ->', 'gen_inval_exception (', 'gen_inval_exception', 'it', '= rpath', 'hwaddr addr', '* dc', ', hwaddr', 'attrs', '< s', 'end_addr', '; qdev_init', '/', 'qdev_init', 'qdev_init (', ') {', '{']


In [10]:
test = pd.read_json("D:/vuln-detection-ml/data/test.jsonl", lines=True)
test["clean"] = test["func"].apply(clean_code)
X_test = vectorizer.transform(test["clean"])
y_test = test["target"]

nb = MultinomialNB(alpha=0.01).fit(X_train, y_train)

for name, model in [("Naive Bayes", nb), ("SVM", svm)]:
    p = model.predict(X_test)
    print(name)
    print(classification_report(y_test, p, digits=3))
    print(confusion_matrix(y_test, p))

Naive Bayes
              precision    recall  f1-score   support

           0      0.651     0.669     0.660      1486
           1      0.592     0.573     0.582      1246

    accuracy                          0.625      2732
   macro avg      0.622     0.621     0.621      2732
weighted avg      0.624     0.625     0.625      2732

[[994 492]
 [532 714]]
SVM
              precision    recall  f1-score   support

           0      0.655     0.623     0.639      1486
           1      0.575     0.608     0.591      1246

    accuracy                          0.616      2732
   macro avg      0.615     0.616     0.615      2732
weighted avg      0.619     0.616     0.617      2732

[[926 560]
 [488 758]]
